# Tutorial 1: Scanning with Flex

Build a small scanner, then explore longest matches, rule priority, source locations, and token lifetimes.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jianwen-zhu/2026-fall-ece467-jzhu/blob/gh-pages/tutorial1.ipynb)

## Setup

Run these setup cells once, then work through the examples in order. In Colab, use a CPU runtime. On your own computer, install Make, a C compiler, and Flex first.

In [ ]:
%%bash
set -euo pipefail
missing=0
for tool in make cc flex; do command -v "$tool" >/dev/null || missing=1; done
if python3 -c 'import google.colab' 2>/dev/null; then
    if [ "$missing" -eq 1 ]; then
        apt-get -qq update
        apt-get -qq install -y build-essential flex
    fi
fi
for tool in make cc flex; do command -v "$tool"; done


## Setup and building

The examples use a local `.tutorial-1-work/` directory. Source files remain available for inspection; the final `make clean` removes generated files only. Rerunning a source cell overwrites that demonstration file, so save any experiments you want to keep first.

`%%bash` runs a cell as a shell script. `%%writefile` writes the remaining cell contents literally to the named file. No Python command wrapper is needed. Shell cells use `set -euo pipefail` so unexpected failures stop the cell rather than being hidden.

The small Makefile below uses the same tools and conventions as the starter: `make` builds, file dependencies determine what needs rebuilding, and `make clean` removes build products. Recipe lines begin with a tab. `$<` names the first prerequisite and `$@` the target. This demonstration has its own Makefile because its standalone `main` is not the starter's compiler driver.

In [ ]:
%%bash
set -euo pipefail
mkdir -p .tutorial-1-work


In [ ]:
%%writefile .tutorial-1-work/Makefile
CC = cc
CFLAGS = -std=c11 -O0 -g -Wall -Wextra
FLEX = flex
.DEFAULT_GOAL := all
.SECONDARY:
.PHONY: all clean
all: words
%.c: %.l
	$(FLEX) -o $@ $<
%: %.c
	$(CC) $(CFLAGS) $< -o $@
clean:
	$(RM) words priority words.c priority.c actual.txt error.txt


## A small scanner

Consider `axis axis2 f32`. In TTL467, `axis` is a keyword, while `axis2` and `f32` are identifiers. The scanner below handles these words and whitespace; it is not a complete TTL467 scanner.

A Flex file has three sections, separated by `%%`: declarations, matching rules, and C helper functions. Each rule associates a pattern with a C action. The location macro records where each match starts; we will examine it below.

`noyywrap` lets this example run without a separate Lex support library. Its `main` function calls the generated scanner.

In [ ]:
%%writefile .tutorial-1-work/words.l
%option noyywrap nounput noinput nodefault
%{
#include <stdio.h>
static size_t line = 1, column = 1, row, col;
#define YY_USER_ACTION \
    row = line; col = column; \
    for (int i = 0; i < yyleng; ++i) { \
        if (yytext[i] == '\n') { ++line; column = 1; } \
        else { ++column; } \
    }
%}
%%
"axis"                  { printf("%zu:%zu KEYWORD %s\n", row, col, yytext); }
[A-Za-z_][A-Za-z0-9_]*    { printf("%zu:%zu IDENTIFIER %s\n", row, col, yytext); }
[ \t\r\n]+              ;
.                       { fprintf(stderr, "%zu:%zu bad: %s\n", row, col, yytext);
                          return 1; }
%%
int main(void) { return yylex(); }


In [ ]:
%%bash
set -euo pipefail
cd .tutorial-1-work
make
printf 'axis axis2 f32\n' | ./words | tee actual.txt
printf '1:1 KEYWORD axis\n1:6 IDENTIFIER axis2\n1:12 IDENTIFIER f32\n' | diff - actual.txt

## Longest match and rule priority

Flex chooses the longest match. If several rules match the same number of characters, it chooses the first rule.

For `axis`, both the keyword and identifier patterns match four characters, so the keyword rule wins. For `axis2`, the identifier pattern matches five characters and wins regardless of rule order. A dtype name such as `f32` is an identifier in TTL467.

The following variant puts identifiers first. Compare its output and generation warning with the original scanner. The original file is unchanged.

In [ ]:
%%writefile .tutorial-1-work/priority.l
%option noyywrap nounput noinput nodefault
%{
#include <stdio.h>
static size_t line = 1, column = 1, row, col;
#define YY_USER_ACTION \
    row = line; col = column; \
    for (int i = 0; i < yyleng; ++i) { \
        if (yytext[i] == '\n') { ++line; column = 1; } \
        else { ++column; } \
    }
%}
%%
[A-Za-z_][A-Za-z0-9_]*    { printf("%zu:%zu IDENTIFIER %s\n", row, col, yytext); }
"axis"                  { printf("%zu:%zu KEYWORD %s\n", row, col, yytext); }
[ \t\r\n]+              ;
.                       { fprintf(stderr, "%zu:%zu bad: %s\n", row, col, yytext);
                          return 1; }
%%
int main(void) { return yylex(); }


In [ ]:
%%bash
set -euo pipefail
cd .tutorial-1-work
make priority
printf 'axis axis2\n' | ./priority | tee actual.txt
printf '1:1 IDENTIFIER axis\n1:6 IDENTIFIER axis2\n' | diff - actual.txt

The keyword rule is now unreachable: every occurrence of `axis` also matches the earlier identifier rule. The warning identifies a real change in token classification. Rule priority does not split `axis2`, because longest match is considered first.

## Source locations and skipped input

Skipping whitespace does not mean ignoring its effect on positions. `YY_USER_ACTION` saves the start of each match before advancing through its characters. It runs for whitespace matches too.

In the next input, `axis` starts at line 1, column 3; `f32` starts at line 2, column 2; and `?` occurs at line 2, column 6. Columns here count source characters, not visual tab stops.

In [ ]:
%%bash
set -euo pipefail
cd .tutorial-1-work
if printf '  axis\n f32 ?\n' | ./words > actual.txt 2> error.txt; then
    echo "Expected an invalid-character error" >&2
    exit 1
fi
cat actual.txt error.txt
printf '1:3 KEYWORD axis\n2:2 IDENTIFIER f32\n' | diff - actual.txt
printf '2:6 bad: ?\n' | diff - error.txt

The scanner reports `?` and returns a nonzero status. Silently skipping an invalid character could leave a plausible but incorrect token stream.

For `axis item := 4;`, token starts are columns 1, 6, 11, 14, and 15. Numeric rules introduce further distinctions: TTL467 treats `-1.0` as two tokens, but `1e-3` as one floating-point token. Our small scanner does not yet implement those rules.

## The TTC scanner interface

The Stage 1 starter uses the same matching mechanism within a supplied C interface:

| This example | TTC starter |
|---|---|
| main calls yylex on stdin | scan receives mapped source |
| actions print | emit appends an owned token |
| local location macro | supplied YY_USER_ACTION tracks locations |
| scanner reaches EOF | wrapper appends an explicit EOF checkpoint token |
| temporary yytext | retained spelling copied into the token arena |

`YY_INPUT` refills Flex's small mutable buffer from mapped source bytes. The mapped file and the scanner buffer serve different purposes; another whole-file copy is unnecessary.

`yytext` refers to scanner-managed storage that can change on later matches. Tokens that retain their spelling need their own copy, owned by the token arena. The supplied wrapper handles this bookkeeping while students implement the lexical rules.

Building the scaffold checks the toolchain, not scanner correctness. The project's `make check` is expected to fail until its rules are implemented.

## Summary

- Longest match determines token boundaries; rule order breaks ties.
- Every consumed character affects source locations, including skipped whitespace.
- Invalid input must produce an error rather than silently disappear.
- Retained token spellings need a lifetime independent of Flex's matching buffer.

For further exploration, consider how a scanner remembers that it is inside a block comment, and how EOF detects an unterminated comment. A character class such as `[/*]` matches one character; it does not match the two-character delimiter `/*`.

## Using the project starter

The same edit-build-run cycle applies to Project Stage 1. In a terminal at the root of your extracted stage package, run:

```sh
make -C 01_scanner
make -C 01_scanner check-starter
make -C 01_scanner check
```

In the course checkout, use `course/project/01_scanner` instead. In Jupyter, prefix a single command with `!`, or place these commands in a `%%bash` cell. Paths are relative to the notebook's working directory.

The stage Makefile includes the supplied `support/project.mk`: do not replace it with the toy Makefile above. `make` builds `ttlc`; `check-starter` checks the **unmodified scaffold**, not a completed solution; `check` checks your implementation and is expected to fail before you implement the stage. After editing your scanner.l, rerun `make` and `make check`. Later, `make -C 01_scanner clean` removes generated files without deleting your source.

The following cell cleans only this notebook's demonstration build products.

In [ ]:
%%bash
set -euo pipefail
cd .tutorial-1-work
make clean
printf 'Tutorial 1 demonstrations passed; generated files removed.\n'